#  Image Classification — Starter Notebook

**Task:** Binary classification (class 0 vs class 1) on the  dataset.


In [1]:
import os
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # must be before torch import
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from model_class import build_model

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
# ── Configuration ──
DATA_ROOT = "dataset/"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")

BATCH_SIZE    = 64
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 0.0
NUM_WORKERS   = 4
IMG_SIZE      = 224

In [ ]:
# ── Transforms ──
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [4]:
# ── Dataset ──
class TrainDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        for label in [0, 1]:
            cls_dir = os.path.join(root, str(label))
            for fname in sorted(os.listdir(cls_dir)):
                if fname.endswith(".png"):
                    self.samples.append((os.path.join(cls_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = TrainDataset(TRAIN_DIR, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
print(f"Train: {len(train_dataset)} images")

Train: 18332 images


In [5]:
# ── Model, Optimizer, Loss ──
model = build_model(num_classes=2).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

Parameters: 11,177,538


In [6]:
# ── Training Loop ──
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = torch.tensor(labels, dtype=torch.long).to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  Loss: {running_loss/total:.4f}  Acc: {correct/total:.4f}")

print("\nTraining complete.")

/tmp/ipykernel_656106/1736195913.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, dtype=torch.long).to(DEVICE)


Epoch 01/20  Loss: 0.3674  Acc: 0.8375
Epoch 02/20  Loss: 0.1379  Acc: 0.9483
Epoch 03/20  Loss: 0.0947  Acc: 0.9638
Epoch 04/20  Loss: 0.0739  Acc: 0.9733
Epoch 05/20  Loss: 0.0583  Acc: 0.9793
Epoch 06/20  Loss: 0.0507  Acc: 0.9818
Epoch 07/20  Loss: 0.0469  Acc: 0.9838
Epoch 08/20  Loss: 0.0389  Acc: 0.9869
Epoch 09/20  Loss: 0.0341  Acc: 0.9877
Epoch 10/20  Loss: 0.0309  Acc: 0.9895
Epoch 11/20  Loss: 0.0274  Acc: 0.9900
Epoch 12/20  Loss: 0.0256  Acc: 0.9915
Epoch 13/20  Loss: 0.0241  Acc: 0.9921
Epoch 14/20  Loss: 0.0244  Acc: 0.9922
Epoch 15/20  Loss: 0.0198  Acc: 0.9924
Epoch 16/20  Loss: 0.0192  Acc: 0.9939
Epoch 17/20  Loss: 0.0164  Acc: 0.9938
Epoch 18/20  Loss: 0.0185  Acc: 0.9935
Epoch 19/20  Loss: 0.0189  Acc: 0.9929
Epoch 20/20  Loss: 0.0135  Acc: 0.9946

Training complete.


In [7]:
# ── Save Model ──
# IMPORTANT: save the FULL model, not just state_dict
torch.save(model, "saved_model.pth")
print(f"Saved to saved_model.pth ({os.path.getsize('saved_model.pth') / 1e6:.1f} MB)")
print("\nSubmit: model_class.py + saved_model.pth")

Saved to saved_model.pth (44.8 MB)

Submit: model_class.py + saved_model.pth
